# Verify full-precision native categorical records and frozen fitting


Verify current inputs/outputs, exact native calls and class-score floating values, training-only fitting and matched baselines. This remains prediction-grade surrogate recovery, not independent biological accuracy.

In [1]:
from pathlib import Path
import json, hashlib, ast
import numpy as np
import pandas as pd
from starplast import artifacts as A, label_records as L, organisms as O, strategies as S
from starplast.splits import read_split
out = Path('results/label_knn_pilot_2026_10_08_v5')
m = json.loads((out / 'manifest.json').read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
for path, digest in m['input_sha256'].items():
    assert sha(path) == digest, path
for name, digest in m['outputs'].items():
    assert sha(out / name) == digest, name
split, exclusions = read_split(out / 'split.json')
raw = json.loads((out / 'held_out/manifest.json').read_text())
spec = A.spec_from_dict(raw['spec'])
artifact = A.read_artifact(out / 'held_out', expected=spec, split=split)
state = artifact.payloads['model_state.json']
frame = pd.read_parquet(O.nodes_path(O.TOXOPLASMA)).set_index('gene_id')
ids = tuple(a.entity for a in split.assignments)
features = frame.loc[list(ids), state['input_columns']]
labels = pd.Series(state['training_labels'], index=state['fit_entities'])
batch = L.feature_knn(features, labels, split=split, exclusions=exclusions, k=state['k'], min_share=state['min_share'])
train_context = S.Context(features.loc[state['fit_entities']].reset_index(drop=True), graph={}, organism=O.TOXOPLASMA)
np.testing.assert_array_equal(state['training_vectors'], train_context.matrix(state['kept_columns']))
native_rows = batch.rows.astype(object).where(batch.rows.notna(), None).to_dict(orient='records')
assert native_rows == [{k: row[k] for k in batch.rows.columns} for row in artifact.payloads['rows.json']]
assert np.array_equal(batch.class_scores.to_numpy(), np.array(artifact.payloads['class_scores.json']['scores'], dtype=float))
assert len(batch.rows) == 560 and not m['summary']['benchmark_admitted']
for path in ('starplast/label_records.py', 'scripts/freeze_label_knn_pilot.py', 'starplast/record_scorecards.py'):
    ast.parse(Path(path).read_text(), feature_version=(3, 10))
{'outer_test_rows': len(batch.rows), 'native_calls_and_score_values': 'exact', 'grade': m['summary']['truth_grade'], 'biological_admission': False, 'training_features': len(state['kept_columns'])}

{'outer_test_rows': 560, 'native_calls_and_score_values': 'exact', 'grade': 'prediction', 'biological_admission': False, 'training_features': 366}

Previous diagnostics retain their own original scripts/code. Tuple/list transport normalization and pandas default float rounding were corrected explicitly. The final artifact records complete pipeline code dependencies and full native float values; metric hashes distinguish a one-ULP score change. Serialization refinements preserved their diagnostic predictions; the current rank-transform adapter is a separate version with its own measurements. Training vectors match the original native rank transform exactly; held-out transforms use frozen training distributions.